# Rubric Agent

This notebook walks through the three-stage rubric pipeline: extracting
requirements from policy text, refining with historical audit findings,
and producing evidence-backed verdicts against a project description.

## Imports

In [ ]:
from agentic_patterns.core.rubric import (
    RubricBuilder,
    RubricEvaluator,
    refine_with_history,
)
from agentic_patterns.core.vectordb import get_vector_db, MultiSourceRetriever
from agentic_patterns.core.vectordb.chunking import chunk_by_paragraphs
from agentic_patterns.core.doc_ingestion.models import DocumentProvenance

## Sample data

Fake SOC 2 policy, audit findings, and a project security description.

In [ ]:
POLICY_TEXT = """\
Access Control Policy
All production systems MUST enforce role-based access control (RBAC).
User accounts MUST be reviewed quarterly and inactive accounts disabled within 30 days.
Privileged access SHOULD require multi-factor authentication at every login.

Encryption Policy
Data at rest MUST be encrypted using AES-256 or equivalent.
Data in transit MUST use TLS 1.2 or higher for all internal and external communications.
Encryption keys SHOULD be rotated annually and MAY be rotated more frequently for sensitive systems.

Logging and Monitoring Policy
All authentication events MUST be logged with timestamp, user ID, and outcome.
Administrative actions MUST be captured in an immutable audit trail.
Anomaly detection SHOULD be enabled on critical systems and alerts reviewed within 24 hours.

Incident Response Policy
A documented incident response plan MUST be maintained and tested annually.
Security incidents MUST be reported to the security team within one hour of detection.
Post-incident reviews SHOULD be completed within five business days.
"""

AUDIT_FINDINGS_TEXT = """\
Q3 Audit Finding: Three service accounts with production access had not been reviewed in over six months.
Q3 Audit Finding: Two internal microservices communicated over plain HTTP instead of TLS.
Q3 Audit Finding: Admin console actions were logged but logs lacked immutability guarantees.

Q1 Audit Finding: Quarterly access review was completed 15 days late for the payments team.
Q1 Audit Finding: Encryption key rotation had not occurred for the analytics database in 18 months.

Q4 Audit Finding: Incident response plan existed but had not been tested since initial creation two years ago.
Q4 Audit Finding: Two critical alerts from the anomaly detection system went unacknowledged for 48 hours.
"""

PROJECT_DESCRIPTION_TEXT = """\
Project Aurora -- Security Posture Summary

Aurora enforces RBAC via AWS IAM with quarterly access reviews automated through a custom script.
MFA is required for all human users but not for CI/CD service accounts.

All databases use AES-256 encryption at rest. Internal service-to-service traffic uses mTLS.
Encryption key rotation is handled by AWS KMS with a 365-day rotation policy.

Authentication events are logged to CloudWatch with structured JSON entries.
Admin actions are captured but stored in the same mutable log stream as application logs.

Aurora has a documented incident response runbook. It was last tested eight months ago.
Anomaly detection is not currently enabled; the team relies on manual dashboard reviews.
"""

## Ingest into vector databases

Create three collections and chunk each text by paragraphs.

In [ ]:
policy_index = get_vector_db("rubric_demo_policy")
history_index = get_vector_db("rubric_demo_history")
project_index = get_vector_db("rubric_demo_project")

for vdb, text, source in [
    (policy_index, POLICY_TEXT, "soc2_policy"),
    (history_index, AUDIT_FINDINGS_TEXT, "audit_findings"),
    (project_index, PROJECT_DESCRIPTION_TEXT, "project_aurora"),
]:
    if vdb.count() == 0:
        chunks = chunk_by_paragraphs(
            text, DocumentProvenance(source=source), min_lines=1
        )
        n = vdb.ingest(chunks)
        print(f"{source}: ingested {n} chunks")
    else:
        print(f"{source}: already populated ({vdb.count()} chunks)")

## Stage 1 -- Build rubric from policy

Extract requirement items (MUST / SHOULD / MAY) from the policy text.

In [ ]:
builder = RubricBuilder()
rubric = await builder.build_from_policy(policy_index, rubric_name="soc2_demo")

In [ ]:
print(f"Rubric: {rubric.rubric_id}  ({len(rubric.items)} items)\n")
for item in rubric.items:
    print(f"[{item.requirement_level.value}] {item.title}  (weight={item.weight})")

## Stage 2 -- Refine with historical findings

Bump weights for items that recur in audit history and promote new concerns.

In [ ]:
rubric_v2 = await refine_with_history(rubric, history_index, policy_index)

In [ ]:
print(
    f"Rubric v{rubric_v2.provenance.get('version', '?')}  ({len(rubric_v2.items)} items)\n"
)
for item in rubric_v2.items:
    print(f"[{item.requirement_level.value}] {item.title}  (weight={item.weight})")

## Stage 3 -- Evidence-backed assessment

Evaluate each rubric item against the project description, policy, and audit history.

In [ ]:
retriever = MultiSourceRetriever(
    {
        "policy": policy_index,
        "history": history_index,
        "project": project_index,
    }
)
evaluator = RubricEvaluator()
verdicts = await evaluator.evaluate(rubric_v2, retriever)

In [ ]:
for v in verdicts:
    print(f"[{v.status.value}] {v.item_id}")
    print(f"  Rationale: {v.rationale[:120]}")
    if v.missing_evidence:
        print(f"  Missing: {', '.join(v.missing_evidence)}")
    print()

## Adversarial simulation (optional)

A `RedTeamAgent` can challenge the verdicts, same pattern as in the debate notebook.
Uncomment the cell below to try it.

In [ ]:
# from agentic_patterns.core.agents import RedTeamAgent
#
# red_team = RedTeamAgent(
#     threat_model="SOC 2 compliance gaps: data exposure, access control bypass, undetected incidents."
# )
# summary = "\n".join(f"[{v.status.value}] {v.item_id}: {v.rationale[:80]}" for v in verdicts)
# rt_result = await red_team.analyze(result=summary, context=PROJECT_DESCRIPTION_TEXT)
#
# for ch in rt_result.challenges:
#     print(f"[{ch.severity}] {ch.claim}")
#     print(f"  Attack: {ch.attack}")
#     print()